In [9]:
import torch as tr
import numpy as np
import pandas as pd
import os 
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import DataLoader
import time
import sys

# --- PROJECT IMPORTS ---
current_dir = os.getcwd() 
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.dataset import SeqDataset, pad_batch 
from src.metrics import contact_f1
from src.utils import save_config, load_model
# --- UTILITIES ---

def get_timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

In [10]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    epoch_loss = []
    
    pbar = tqdm(loader, desc="Training", leave=False)
    
    for batch in pbar:
        cond = batch["outer"].to(device)
        target = batch["contact_oh"].to(device)
        mask = batch["mask"].to(device)       
        
        optimizer.zero_grad()
        
        # Forward pass (diffusion loss)
        loss = model.forward_all_timesteps(target, cond, mask=mask)
             
        loss.backward()
        optimizer.step()
        
        epoch_loss.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    return np.mean(epoch_loss)

@tr.no_grad()
def validate(model, loader, device):
    """
    Computes Validation Loss and F1 Score (via sampling).
    """
    model.eval()
    val_loss = []
    val_f1 = []
    
    for batch in tqdm(loader, desc="Validating", leave=False):
        cond = batch["outer"].to(device)
        target = batch["contact_oh"].to(device)
        mask = batch["mask"].to(device)       
        lens = batch["length"]
        
        # 1. Validation Loss (No sampling)
        loss = model.forward_all_timesteps(target, cond, mask=mask)
        val_loss.append(loss.item())
        
        # 2. Validation F1 (Sampling required)
        samples = model._sample(cond)
        f1_score = contact_f1(samples, target, lengths=lens, reduce=True)
        val_f1.append(f1_score)
        
    return np.mean(val_loss), np.mean(val_f1)


# Profiler

In [11]:
from torch.profiler import profile, ProfilerActivity, schedule, tensorboard_trace_handler, record_function

In [12]:
def _iter_loader(loader, max_batches=None):
    if max_batches is None:
        yield from loader
        return

    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        yield batch


def train_one_epoch_profiled(model, loader, optimizer, device, prof=None, max_batches=None):
    model.train()
    for batch in _iter_loader(loader, max_batches=max_batches):
        with record_function("train_batch"):
            cond = batch["outer"].to(device, non_blocking=True)
            target = batch["contact_oh"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with record_function("train_forward"):
                loss = model.forward_all_timesteps(target, cond, mask=mask)
            with record_function("train_backward"):
                loss.backward()
            with record_function("train_step"):
                optimizer.step()

        if prof is not None:
            prof.step()


@tr.no_grad()
def eval_one_epoch_profiled(model, loader, device, compute_f1=True, prof=None, max_batches=None):
    model.eval()
    for batch in _iter_loader(loader, max_batches=max_batches):
        with record_function("eval_batch"):
            cond = batch["outer"].to(device, non_blocking=True)
            target = batch["contact_oh"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)

            with record_function("eval_loss_forward"):
                _ = model.forward_all_timesteps(target, cond, mask=mask)

            if compute_f1:
                lens = batch["length"]
                with record_function("eval_sample"):
                    samples = model._sample(cond)
                with record_function("eval_contact_f1"):
                    _ = contact_f1(samples, target, lengths=lens, reduce=True)

        if prof is not None:
            prof.step()

In [15]:
def _safe_total_ms(key_avgs, name, use_cuda=False, self_only=True):
    for evt in key_avgs:
        if evt.key == name:
            if use_cuda:
                t = evt.self_cuda_time_total if self_only else evt.cuda_time_total
            else:
                t = evt.self_cpu_time_total if self_only else evt.cpu_time_total
            return t / 1000.0
    return 0.0


def run_profiler(config):
    """
    Profiles train + eval(no F1) + eval(with F1) in one run and reports F1 overhead.
    """
    timestamp = get_timestamp()
    exp_name = f"exp_T{config['timesteps']}_E{config['epochs']}_{timestamp}"
    log_path = config["log_path"]
    log_dir = log_path
    os.makedirs(log_dir, exist_ok=True)

    save_config(config, log_dir)

    print(f"\n{'='*40}")
    print(f"STARTING EXPERIMENT: {exp_name}")
    print(f"Config: {config}")
    print(f"{'='*40}")

    device = tr.device("cuda" if tr.cuda.is_available() else "cpu")

    train_ds = SeqDataset(config["train_path"])
    val_ds = SeqDataset(config["val_path"])

    use_cuda = tr.cuda.is_available()
    train_loader = DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True,
        collate_fn=pad_batch,
        num_workers=2,
        pin_memory=use_cuda,
        persistent_workers=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config["batch_size"],
        shuffle=False,
        collate_fn=pad_batch,
        num_workers=2,
        pin_memory=use_cuda,
        persistent_workers=True,
    )
     
    model = load_model(config=config, eval=False)
    optimizer = tr.optim.Adam(model.parameters(), lr=config["lr"])

    trace_dir = os.path.join(log_dir, "profiler_traces")
    os.makedirs(trace_dir, exist_ok=True)

    profile_batches = config.get("profile_batches", 5) 
    activities = [ProfilerActivity.CPU]
    if use_cuda:
        activities.append(ProfilerActivity.CUDA)
    steps_per_epoch = 3 * profile_batches
    with profile(
        activities=activities,
        schedule=schedule(wait=0, warmup=2, active=steps_per_epoch, repeat=1),
        on_trace_ready=tensorboard_trace_handler(trace_dir),
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
    ) as prof:
        for epoch in range(1, config["epochs"] + 1):
            print(f"\nEpoch {epoch}/{config['epochs']}")

            with record_function("epoch_train"):
                train_one_epoch_profiled(
                    model, train_loader, optimizer, device, prof=prof, max_batches=profile_batches
                )

            with record_function("epoch_eval_no_f1"):
                eval_one_epoch_profiled(
                    model, val_loader, device, compute_f1=False, prof=prof, max_batches=profile_batches
                )

            with record_function("epoch_eval_with_f1"):
                eval_one_epoch_profiled(
                    model, val_loader, device, compute_f1=True, prof=prof, max_batches=profile_batches
                )

    sort_key = "self_cuda_time_total" if use_cuda else "self_cpu_time_total"
    print("\nTop profiler ops:")
    print(prof.key_averages().table(sort_by=sort_key, row_limit=30))

    events = prof.key_averages()
    eval_no_f1_ms = _safe_total_ms(events, "epoch_eval_no_f1")
    eval_with_f1_ms = _safe_total_ms(events, "epoch_eval_with_f1")
    f1_only_ms = _safe_total_ms(events, "eval_contact_f1")
    sample_ms = _safe_total_ms(events, "eval_sample")

    if eval_no_f1_ms > 0:
        overhead_pct = ((eval_with_f1_ms - eval_no_f1_ms) / eval_no_f1_ms) * 100.0
    else:
        overhead_pct = 0.0

    print("\nProfiler summary (CPU self time):")
    print(f"- eval_no_f1:   {eval_no_f1_ms:.2f} ms")
    print(f"- eval_with_f1: {eval_with_f1_ms:.2f} ms")
    print(f"- eval_sample:  {sample_ms:.2f} ms")
    print(f"- contact_f1:   {f1_only_ms:.2f} ms")
    print(f"- F1 overhead:  {overhead_pct:.2f}%")

    print(f"Experiment profiled. Traces saved to: {trace_dir}")
    return log_dir

In [17]:
BASE_DATA_DIR = "../data/simfolds/simfolds_max128/joined/"

sim = "sim90"
DATA_DIR = os.path.join(BASE_DATA_DIR, sim)
os.makedirs(DATA_DIR, exist_ok=True)

conf = {
    "train_path": f"{DATA_DIR}/train.csv",
    "val_path": f"{DATA_DIR}/valid.csv",
    "log_path": f"logs/",
    "batch_size": 4,
    "lr": 1e-3,
    "epochs": 3,
    "timesteps": 10,
    "profile_batches": 20,
    "note": f"simfold profiler: {sim}. Archive II max 128.",
}
run_profiler(conf)


STARTING EXPERIMENT: exp_T10_E3_20260225_151518
Config: {'train_path': '../data/simfolds/simfolds_max128/joined/sim90/train.csv', 'val_path': '../data/simfolds/simfolds_max128/joined/sim90/valid.csv', 'log_path': 'logs/', 'batch_size': 4, 'lr': 0.001, 'epochs': 3, 'timesteps': 10, 'profile_batches': 20, 'note': 'simfold profiler: sim90. Archive II max 128.'}

Epoch 1/3


STAGE:2026-02-25 15:15:19 3389960:3389960 ActivityProfilerController.cpp:312] Completed Stage: Warm Up



Epoch 2/3


[W CPUAllocator.cpp:235] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event
STAGE:2026-02-25 15:15:32 3389960:3389960 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-02-25 15:15:32 3389960:3389960 ActivityProfilerController.cpp:322] Completed Stage: Post Processing



Epoch 3/3

Top profiler ops:
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             aten::convolution_backward         4.41%     568.025ms         7.01%     902.623ms     153.507us        1.725s        29.79%        1.879s     319.500us           0 b           

'logs/'

## MLP simple para entender `torch.profiler`

Este bloque agrega un experimento chico y controlado para medir **tiempo** y **memoria** con profiler.

Objetivo:
- Ver que parte del paso de entrenamiento consume mas tiempo (`forward`, `backward`, `step`).
- Medir impacto en memoria al cambiar `batch_size`, `hidden_dim` y `num_layers`.
- Guardar traces para abrir en TensorBoard.

### Conceptos clave del profiler

- `schedule(wait, warmup, active)`:
  - `wait`: pasos ignorados (arranque).
  - `warmup`: pasos de calentamiento (sin guardar trace final).
  - `active`: pasos que si se guardan en la traza.
- `prof.step()`: avanza el reloj del profiler en cada batch. Sin esto, el schedule no progresa.
- `record_function('nombre')`: pone etiquetas visibles en TensorBoard para entender fases del entrenamiento.
- `profile_memory=True`: habilita metricas de memoria (CPU/GPU) por operador.
- `tensorboard_trace_handler(ruta)`: escribe archivos `.pt.trace.json` para visualizarlos.

In [2]:
import os
import time
import torch as tr
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.profiler import profile, ProfilerActivity, schedule, tensorboard_trace_handler, record_function

### Modelo y utilidades de memoria

Estas funciones definen un MLP basico y calculan memoria aproximada de:
- parametros del modelo
- estados del optimizador
- pico de memoria CUDA (si hay GPU)

In [3]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout=0.0):
        super().__init__()
        layers = []
        d_in = input_dim
        for _ in range(num_layers):
            layers.append(nn.Linear(d_in, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d_in = hidden_dim
        layers.append(nn.Linear(d_in, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def _bytes_to_mb(num_bytes):
    return num_bytes / (1024 ** 2)


def _model_param_mb(model):
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return _bytes_to_mb(total_bytes)


def _optimizer_state_mb(optimizer):
    total_bytes = 0
    for state in optimizer.state.values():
        for value in state.values():
            if tr.is_tensor(value):
                total_bytes += value.numel() * value.element_size()
    return _bytes_to_mb(total_bytes)

### Datos sinteticos y configuracion del experimento

Usamos datos aleatorios para aislar el costo del modelo/entrenamiento y no mezclarlo con I/O real.

In [4]:
def make_synth_loader(num_samples, input_dim, output_dim, batch_size, use_cuda, seed=0):
    tr.manual_seed(seed)
    x = tr.randn(num_samples, input_dim)
    y = tr.randn(num_samples, output_dim)
    ds = TensorDataset(x, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=use_cuda)


MLP_PROFILE_CFG = {
    'log_root': 'logs/mlp_profiler',
    'seed': 123,
    'num_samples': 8192,
    'input_dim': 1024,
    'hidden_dim': 2048,
    'output_dim': 128,
    'num_layers': 3,
    'dropout': 0.1,
    'batch_size': 64,
    'lr': 1e-3,
    # schedule del profiler
    'wait': 1,
    'warmup': 1,
    'active': 20,
    'tag': 'mlp_baseline',
}

### Funcion principal de profiling

Que mide esta funcion:
- Tiempo por bloques (`mlp_forward`, `mlp_backward`, `mlp_step`) en la traza.
- Memoria peak CUDA (si hay GPU).
- Memoria aproximada de parametros y estado del optimizador.

Que guardaras para TensorBoard:
- `logs/mlp_profiler/<tag>/profiler_traces/*.pt.trace.json`

In [5]:
def run_mlp_profile(cfg):
    device = tr.device('cuda' if tr.cuda.is_available() else 'cpu')
    use_cuda = device.type == 'cuda'

    loader = make_synth_loader(
        num_samples=cfg['num_samples'],
        input_dim=cfg['input_dim'],
        output_dim=cfg['output_dim'],
        batch_size=cfg['batch_size'],
        use_cuda=use_cuda,
        seed=cfg.get('seed', 0),
    )

    model = SimpleMLP(
        input_dim=cfg['input_dim'],
        hidden_dim=cfg['hidden_dim'],
        output_dim=cfg['output_dim'],
        num_layers=cfg['num_layers'],
        dropout=cfg.get('dropout', 0.0),
    ).to(device)
    opt = tr.optim.Adam(model.parameters(), lr=cfg['lr'])
    criterion = nn.MSELoss()

    run_tag = cfg.get('tag', f"mlp_{int(time.time())}")
    trace_dir = os.path.join(cfg['log_root'], run_tag, 'profiler_traces')
    os.makedirs(trace_dir, exist_ok=True)

    activities = [ProfilerActivity.CPU]
    if use_cuda:
        activities.append(ProfilerActivity.CUDA)
        tr.cuda.reset_peak_memory_stats()

    total_steps = cfg['wait'] + cfg['warmup'] + cfg['active']
    losses = []

    with profile(
        activities=activities,
        schedule=schedule(wait=cfg['wait'], warmup=cfg['warmup'], active=cfg['active'], repeat=1),
        on_trace_ready=tensorboard_trace_handler(trace_dir),
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        step = 0
        while step < total_steps:
            for xb, yb in loader:
                if step >= total_steps:
                    break

                with record_function('mlp_train_batch'):
                    xb = xb.to(device, non_blocking=use_cuda)
                    yb = yb.to(device, non_blocking=use_cuda)

                    with record_function('mlp_forward'):
                        pred = model(xb)
                        loss = criterion(pred, yb)

                    with record_function('mlp_backward'):
                        opt.zero_grad(set_to_none=True)
                        loss.backward()

                    with record_function('mlp_step'):
                        opt.step()

                losses.append(loss.item())
                prof.step()
                step += 1

    sort_key = 'self_cuda_time_total' if use_cuda else 'self_cpu_time_total'
    print(prof.key_averages().table(sort_by=sort_key, row_limit=20))

    result = {
        'tag': run_tag,
        'trace_dir': trace_dir,
        'mean_loss': sum(losses) / max(len(losses), 1),
        'param_mb': _model_param_mb(model),
        'optim_state_mb': _optimizer_state_mb(opt),
        'peak_cuda_mb': _bytes_to_mb(tr.cuda.max_memory_allocated()) if use_cuda else 0.0,
        'device': str(device),
        'batch_size': cfg['batch_size'],
        'input_dim': cfg['input_dim'],
        'hidden_dim': cfg['hidden_dim'],
        'num_layers': cfg['num_layers'],
    }

    print('Run:', result['tag'])
    print('Trace dir:', result['trace_dir'])
    print('Params (MB):', f"{result['param_mb']:.2f}")
    print('Optimizer state (MB):', f"{result['optim_state_mb']:.2f}")
    if use_cuda:
        print('Peak CUDA allocated (MB):', f"{result['peak_cuda_mb']:.2f}")

    return result

### Corrida base

Ejecuta una corrida para validar que se crean traces y ver una primera estimacion de memoria.

In [6]:
baseline_result = run_mlp_profile(MLP_PROFILE_CFG)
baseline_result

STAGE:2026-02-25 15:34:53 3491904:3491904 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2026-02-25 15:34:53 3491904:3491904 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-02-25 15:34:53 3491904:3491904 ActivityProfilerController.cpp:322] Completed Stage: Post Processing


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                aten::_foreach_addcdiv_         0.24%     256.000us         0.34%     355.000us      17.750us       5.074ms        15.43%       5.074ms     253.700us           0 b           0 b           0 b           0 

{'tag': 'mlp_baseline',
 'trace_dir': 'logs/mlp_profiler/mlp_baseline/profiler_traces',
 'mean_loss': 1.0077628086913715,
 'param_mb': 41.02392578125,
 'optim_state_mb': 82.04788208007812,
 'peak_cuda_mb': 221.71337890625,
 'device': 'cuda',
 'batch_size': 64,
 'input_dim': 1024,
 'hidden_dim': 2048,
 'num_layers': 3}

### Sweep de memoria

Esta celda cambia tamano del modelo y batch para comparar consumo:
- `hidden_dim` y `num_layers` aumentan parametros y activaciones.
- `batch_size` impacta fuerte en activaciones y pico de memoria durante backward.

In [7]:
memory_sweep = [
    {'tag': 'mlp_small', 'hidden_dim': 512, 'num_layers': 2, 'batch_size': 32},
    {'tag': 'mlp_medium', 'hidden_dim': 1024, 'num_layers': 3, 'batch_size': 64},
    {'tag': 'mlp_large', 'hidden_dim': 4096, 'num_layers': 4, 'batch_size': 128},
]

sweep_results = []
for variant in memory_sweep:
    cfg = dict(MLP_PROFILE_CFG)
    cfg.update(variant)
    cfg['active'] = 12
    sweep_results.append(run_mlp_profile(cfg))

sweep_results

STAGE:2026-02-25 15:36:02 3491904:3491904 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2026-02-25 15:36:02 3491904:3491904 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-02-25 15:36:02 3491904:3491904 ActivityProfilerController.cpp:322] Completed Stage: Post Processing


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                            aten::addmm         2.88%       1.121ms         3.96%       1.541ms      42.806us     396.000us        14.27%     396.000us      11.000us           0 b           0 b       1.69 Mb       1.69 M

STAGE:2026-02-25 15:36:03 3491904:3491904 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2026-02-25 15:36:03 3491904:3491904 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-02-25 15:36:03 3491904:3491904 ActivityProfilerController.cpp:322] Completed Stage: Post Processing


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         1.05%       1.443ms         1.50%       2.061ms      24.536us       1.166ms        15.89%       1.194ms      14.214us           0 b           0 b     159.00 Mb     159.00 M

STAGE:2026-02-25 15:36:04 3491904:3491904 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2026-02-25 15:36:05 3491904:3491904 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2026-02-25 15:36:05 3491904:3491904 ActivityProfilerController.cpp:322] Completed Stage: Post Processing


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         0.67%       1.828ms         0.97%       2.628ms      24.333us      19.882ms        18.72%      22.505ms     208.380us           0 b           0 b       2.55 Gb       2.55 G

[{'tag': 'mlp_small',
  'trace_dir': 'logs/mlp_profiler/mlp_small/profiler_traces',
  'mean_loss': 1.0072874810014452,
  'param_mb': 3.25439453125,
  'optim_state_mb': 6.508811950683594,
  'peak_cuda_mb': 32.69384765625,
  'device': 'cuda',
  'batch_size': 32,
  'input_dim': 1024,
  'hidden_dim': 512,
  'num_layers': 2},
 {'tag': 'mlp_medium',
  'trace_dir': 'logs/mlp_profiler/mlp_medium/profiler_traces',
  'mean_loss': 1.0034991673060827,
  'param_mb': 12.51220703125,
  'optim_state_mb': 25.024444580078125,
  'peak_cuda_mb': 79.15478515625,
  'device': 'cuda',
  'batch_size': 64,
  'input_dim': 1024,
  'hidden_dim': 1024,
  'num_layers': 3},
 {'tag': 'mlp_large',
  'trace_dir': 'logs/mlp_profiler/mlp_large/profiler_traces',
  'mean_loss': 1.0705336758068629,
  'param_mb': 210.06298828125,
  'optim_state_mb': 420.12601470947266,
  'peak_cuda_mb': 1067.25244140625,
  'device': 'cuda',
  'batch_size': 128,
  'input_dim': 1024,
  'hidden_dim': 4096,
  'num_layers': 4}]

### Tabla comparativa

Si hay `pandas`, muestra una tabla limpia con las metricas principales de memoria y la ruta de trace.

In [8]:
try:
    import pandas as pd
    df = pd.DataFrame(sweep_results)
    display(df[['tag', 'device', 'batch_size', 'hidden_dim', 'num_layers', 'param_mb', 'optim_state_mb', 'peak_cuda_mb', 'mean_loss', 'trace_dir']])
except Exception:
    for row in sweep_results:
        print(row)

,tag,device,batch_size,hidden_dim,num_layers,param_mb,optim_state_mb,peak_cuda_mb,mean_loss,trace_dir
0,mlp_small,cuda,32,512,2,3.254395,6.508812,32.693848,1.007287,logs/mlp_profiler/mlp_small/profiler_traces
1,mlp_medium,cuda,64,1024,3,12.512207,25.024445,79.154785,1.003499,logs/mlp_profiler/mlp_medium/profiler_traces
2,mlp_large,cuda,128,4096,4,210.062988,420.126015,1067.252441,1.070534,logs/mlp_profiler/mlp_large/profiler_traces


### Como abrir en TensorBoard

En Colab/Jupyter:
```python
%load_ext tensorboard
%tensorboard --logdir logs/mlp_profiler
```

En la pestaña *Profile* vas a ver:
- Timeline con `mlp_forward`, `mlp_backward`, `mlp_step`.
- Uso de memoria por operador.
- Diferencias entre corridas (`mlp_small`, `mlp_medium`, `mlp_large`).